# Iterative Prediction — Exploration & Sanity Check

**Goals:**
1. Define a unified iterative prediction interface (baseline + ML models).
2. Sanity check: baseline iterative == baseline linear projection (must match).
3. Run all 5 joblib models iteratively and compare their outputs.
4. Confirm that `dist_t2` (current bank position) updates correctly between steps.

**Key insight:**  
For the baseline model (constant velocity), iterative and linear give identical results — this is the sanity check.  
For ML models (LightGBM, Ridge), velocity can change year-to-year as `dist_t2` updates, so iterative matters.

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path.cwd()
if (_cwd / 'src').exists():
    _backend = _cwd
elif (_cwd.parent / 'src').exists():
    _backend = _cwd.parent
elif (_cwd.parent.parent / 'src').exists():
    _backend = _cwd.parent.parent
else:
    _backend = _cwd
os.chdir(_backend)
sys.path.insert(0, str(_backend))
print('cwd:', os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import src.paths as PATHS
import src.model.baseline_model as BM
import src.data.config as DATA_CONFIG
import src.constants as CONST
from src.model.export_utils import load_model_bundle, predict as bundle_predict

DATA_DIR = PATHS.DATA_DIR

## Config

Set prediction range and step here. These drive all cells below.

In [ ]:
START_YEAR = 2026
END_YEAR   = 2035   # default: START_YEAR + 9
STEP       = 1      # 1 = every year; 3 = 2026, 2029, 2032, 2035; etc.

PREDICTION_YEARS = list(range(START_YEAR, END_YEAR + 1, STEP))
print(f'Prediction years ({len(PREDICTION_YEARS)}): {PREDICTION_YEARS}')

## 1. Load data

In [ ]:
# region_features: ML model input (one row per location, features like v_train, dist_t2, ...)
features_df = pd.read_parquet(DATA_DIR / '02_processed/erosion/region_features.parquet')
print(f'region_features: {len(features_df):,} locations, columns: {list(features_df.columns)}')

In [ ]:
# region_inference_only: last observed dist + year + velocity per location
# This covers locations outside the train/test split that still need predictions
inference_df = pd.read_parquet(DATA_DIR / '02_processed/erosion/region_inference_only.parquet')
print(f'region_inference_only: {len(inference_df):,} locations')
print(inference_df.head(3))

In [ ]:
# Load t3 (most recent observation year per location) from region_split
region_split = pd.read_parquet(DATA_DIR / '02_processed/erosion/region_split.parquet')

# Build a unified starting-point lookup: {loc_id: (last_dist, last_year, v_hist)}
# Priority: features_df (dist_t3, t3 year, v_train), then inference_df (last_dist, last_year, v_last)
start_points = {}

for loc_id, row in features_df.iterrows():
    t3_year = int(region_split.loc[loc_id, 't3']) if loc_id in region_split.index else 2025
    start_points[loc_id] = {
        'last_dist': row['dist_t3'],
        'last_year': t3_year,
        'v_hist': row['v_train'],
    }

for loc_id, row in inference_df.iterrows():
    if loc_id not in start_points:
        start_points[loc_id] = {
            'last_dist': row['last_dist'],
            'last_year': int(row['last_year']),
            'v_hist': row['v_last'],
        }

print(f'Total locations with starting point: {len(start_points):,}')
last_years = pd.Series({k: v['last_year'] for k, v in start_points.items()})
print('last_year distribution:', last_years.value_counts().to_dict())

## 2. Load models

In [ ]:
# Baseline model (pkl): {location_id: mean_velocity}
MODEL_PKL = DATA_DIR / '04_model_outputs/20260217/baseline_model_full_dataset.pkl'
baseline = BM.BaselineErosionModel.load_model(MODEL_PKL)
model_velocities = baseline.model   # dict {loc_id: velocity}
print(f'Baseline model: {len(model_velocities):,} locations')

In [ ]:
# ML models (joblib bundle)
BUNDLE_DIR = DATA_DIR / '04_model_outputs/20260312'
bundle = load_model_bundle(BUNDLE_DIR)
config = bundle['config']
ML_MODELS = ['ols', 'ridge_num', 'ridge_cat', 'lgb']
print('Bundle loaded. Available models:', ML_MODELS)
print('Target:', config['TARGET'])
print('LGB features:', config['FEATS_LGB'])

## 3. Iterative prediction functions

### 3a. Baseline (constant velocity)

For the baseline model, velocity is constant per location.  
- **Pre-step**: advance from `last_year` to `START_YEAR` using historical velocity.
- **Main loop**: for each prediction year, add `velocity × step` to the running dist.

Since velocity is constant, this is mathematically equivalent to the linear formula:  
`dist(y) = last_dist + velocity × (y - last_year)`.

In [ ]:
def predict_iterative_baseline(
    model_velocities: dict,
    start_points: dict,
    start_year: int = START_YEAR,
    end_year: int = END_YEAR,
    step: int = STEP,
) -> pd.DataFrame:
    """
    Iterative prediction using the baseline model (constant velocity per location).

    Each year: dist += velocity * step.
    Pre-step extrapolates from last_year to start_year before the main loop.
    """
    prediction_years = range(start_year, end_year + 1, step)
    rows = []

    for loc_id, velocity in model_velocities.items():
        if loc_id not in start_points:
            continue
        sp = start_points[loc_id]
        last_dist, last_year = sp['last_dist'], sp['last_year']

        # Pre-step: advance from last_year to start_year
        current_dist = (
            last_dist + velocity * (start_year - last_year)
            if last_year < start_year else last_dist
        )

        for year in prediction_years:
            rows.append({
                'location_id': loc_id,
                'year': year,
                'predicted_dist_m': current_dist,
                'velocity_m_per_yr': velocity,
            })
            current_dist += velocity * step

    return pd.DataFrame(rows)

### 3b. ML models (velocity updates with dist_t2)

For ML models, `dist_t2` in the feature matrix is updated each year to the previous year's predicted position.  
All other features (discharge, vegetation, curvature, etc.) remain static.

This allows velocity to change year-to-year as the bank position shifts.

In [ ]:
def predict_iterative_ml(
    bundle: dict,
    features_df: pd.DataFrame,
    start_points: dict,
    model_name: str,
    start_year: int = START_YEAR,
    end_year: int = END_YEAR,
    step: int = STEP,
    dist_feature: str = 'dist_t2',   # feature column to update each iteration
) -> pd.DataFrame:
    """
    Iterative prediction using an ML model from the bundle.

    Each year:
      1. Predict velocity using current features (with updated dist_feature).
      2. Advance dist by velocity * step.
      3. Update dist_feature for the next iteration.

    Only locations present in both features_df and start_points are predicted.
    """
    prediction_years = list(range(start_year, end_year + 1, step))

    # Working copy of features — only predict for locations we have start points for
    locs = [loc for loc in features_df.index if loc in start_points]
    features_iter = features_df.loc[locs].copy()

    # Initialise current_dist and set dist_feature to last observed position
    current_dist = {}
    for loc_id in locs:
        sp = start_points[loc_id]
        last_dist, last_year, v_hist = sp['last_dist'], sp['last_year'], sp['v_hist']

        # Pre-step: advance to start_year using historical velocity
        dist_at_start = (
            last_dist + (v_hist or 0.0) * (start_year - last_year)
            if last_year < start_year else last_dist
        )
        current_dist[loc_id] = dist_at_start
        features_iter.loc[loc_id, dist_feature] = dist_at_start

    rows = []
    for year in prediction_years:
        velocities = bundle_predict(bundle, features_iter, model_name)

        for i, loc_id in enumerate(locs):
            vel = float(velocities[i])
            new_dist = current_dist[loc_id] + vel * step
            rows.append({
                'location_id': loc_id,
                'year': year,
                'predicted_dist_m': new_dist,
                'velocity_m_per_yr': vel,
                'model': model_name,
            })
            current_dist[loc_id] = new_dist
            features_iter.loc[loc_id, dist_feature] = new_dist

    return pd.DataFrame(rows)

## 4. Sanity check: baseline iterative == linear projection

For constant velocity: `dist_iterative(y) = last_dist + v × (y - last_year)` exactly.  
If this assertion fails, there is a bug.

In [ ]:
# --- Run baseline iterative ---
df_iterative = predict_iterative_baseline(model_velocities, start_points)
print(f'Iterative: {len(df_iterative):,} rows, {df_iterative["location_id"].nunique():,} locations')
df_iterative.head()

In [ ]:
# --- Run baseline linear (current pipeline approach) ---
REFERENCE_YEAR = START_YEAR - 1  # = 2025 by default

rows_linear = []
for loc_id, velocity in model_velocities.items():
    if loc_id not in start_points:
        continue
    sp = start_points[loc_id]
    last_dist, last_year = sp['last_dist'], sp['last_year']

    # Extrapolate to reference year
    dist_at_ref = (
        last_dist + velocity * (REFERENCE_YEAR - last_year)
        if last_year < REFERENCE_YEAR else last_dist
    )
    for year in PREDICTION_YEARS:
        rows_linear.append({
            'location_id': loc_id,
            'year': year,
            'predicted_dist_m': dist_at_ref + velocity * (year - REFERENCE_YEAR),
            'velocity_m_per_yr': velocity,
        })

df_linear = pd.DataFrame(rows_linear)
print(f'Linear: {len(df_linear):,} rows, {df_linear["location_id"].nunique():,} locations')

In [ ]:
# --- Compare: should be identical (within floating point) ---
merged = df_iterative.merge(
    df_linear.rename(columns={'predicted_dist_m': 'dist_linear'}),
    on=['location_id', 'year'],
)
merged['diff'] = (merged['predicted_dist_m'] - merged['dist_linear']).abs()

max_diff = merged['diff'].max()
print(f'Max absolute difference (iterative vs linear): {max_diff:.2e}')
assert max_diff < 1e-9, f'SANITY CHECK FAILED: max diff = {max_diff}'
print('✓ Sanity check passed: baseline iterative == linear projection')

## 5. ML model iterative predictions

Run all 4 ML models iteratively. Each year, `dist_t2` is updated to the previous predicted position.  
For `ols` (v_train only feature), velocity is constant — result should be similar to baseline.

In [ ]:
results = {'baseline': df_iterative.assign(model='baseline')}

for model_name in ML_MODELS:
    print(f'Running {model_name}...')
    df_ml = predict_iterative_ml(bundle, features_df, start_points, model_name)
    results[model_name] = df_ml
    print(f'  {len(df_ml):,} rows, {df_ml["location_id"].nunique():,} locations')

print('Done.')

In [ ]:
# Summary: predicted velocity distribution per model
summary_rows = []
for name, df in results.items():
    first_year = df[df['year'] == START_YEAR]
    summary_rows.append({
        'model': name,
        'n_locations': df['location_id'].nunique(),
        'median_velocity': first_year['velocity_m_per_yr'].median(),
        'mean_velocity': first_year['velocity_m_per_yr'].mean(),
        'p90_velocity': first_year['velocity_m_per_yr'].quantile(0.9),
    })

pd.DataFrame(summary_rows).set_index('model').round(3)

## 6. Visual comparison: predicted dist over time (sample locations)

In [ ]:
# Pick a few representative locations that all models have predictions for
common_locs = set(results['baseline']['location_id'])
for df in results.values():
    common_locs &= set(df['location_id'])

# Sample across clusters
from src.erosion.centerline_utils import pick_across_clusters
sample_locs = pick_across_clusters(sorted(common_locs), n=4)
print('Sample locations:', sample_locs)

In [ ]:
MODEL_COLORS = {
    'baseline':   '#1f77b4',
    'ols':        '#ff7f0e',
    'ridge_num':  '#2ca02c',
    'ridge_cat':  '#d62728',
    'lgb':        '#9467bd',
}

fig, axes = plt.subplots(1, len(sample_locs), figsize=(5 * len(sample_locs), 5), constrained_layout=True)
if len(sample_locs) == 1:
    axes = [axes]

for ax, loc_id in zip(axes, sample_locs):
    # Historical starting point
    sp = start_points.get(loc_id, {})
    if sp:
        ax.scatter([sp['last_year']], [sp['last_dist']], color='black', zorder=5,
                   label='last observed', s=60)

    for name, df in results.items():
        sub = df[df['location_id'] == loc_id].sort_values('year')
        if sub.empty:
            continue
        ax.plot(sub['year'], sub['predicted_dist_m'],
                label=name, color=MODEL_COLORS.get(name), lw=1.8,
                ls='--' if name == 'baseline' else '-')

    ax.set_title(loc_id, fontsize=9)
    ax.set_xlabel('Year')
    ax.set_ylabel('Predicted dist from CL (m)')
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8, loc='best')
fig.suptitle('Iterative prediction: all models vs baseline', fontsize=11)
plt.show()

## 7. Check: does dist_t2 update cause velocity drift in ML models?

For the LGB model, plot predicted velocity over time for the same sample locations.  
Baseline velocity is constant; LGB velocity should vary if dist_t2 is a meaningful predictor.

In [ ]:
fig, axes = plt.subplots(1, len(sample_locs), figsize=(5 * len(sample_locs), 4), constrained_layout=True)
if len(sample_locs) == 1:
    axes = [axes]

for ax, loc_id in zip(axes, sample_locs):
    for name in ['baseline', 'lgb', 'ridge_num']:
        df = results.get(name)
        if df is None:
            continue
        sub = df[df['location_id'] == loc_id].sort_values('year')
        if sub.empty:
            continue
        ax.plot(sub['year'], sub['velocity_m_per_yr'],
                label=name, color=MODEL_COLORS.get(name), lw=1.8)

    ax.axhline(0, color='grey', lw=0.8, ls=':')
    ax.set_title(loc_id, fontsize=9)
    ax.set_xlabel('Year')
    ax.set_ylabel('Velocity (m/yr)')
    ax.grid(True, alpha=0.3)

axes[0].legend(fontsize=8)
fig.suptitle('Predicted velocity per year: does it drift?', fontsize=11)
plt.show()